# 02 - CUB: dataset characterization & training validation

CBM + MCBM. **Part A** = full CUB-200 (10 slots). **Part B** = CUB70
occlusion / relabeling axis (~8 slots) -- the cap is raised because CUB70
is a separate analysis axis (visibility, not just training curves).

> **How to use.** Each cell loads an artifact produced by the training / data
> scripts (see `curated/README.md`). Before those run, cells print a `[pending]`
> note instead of failing, so the notebook always executes end to end. On adroit,
> after training, every figure/table fills in and is paper-ready (saved to
> `curated/notebooks/figures/`).

In [ ]:
# Setup -- run on the cluster where CURATED_DATA and the run outputs exist.
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd()
while not (REPO / 'curated').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from curated.analysis import plotting, occlusion, io
plotting.set_paper_style()

DATA = Path(os.environ.get('CURATED_DATA', REPO / 'curated' / '_demo_data'))
RUNS = DATA / 'runs'

def maybe(path, loader=pd.read_parquet):
    '''Load an artifact or return None with a clear note, so the notebook',
    runs top-to-bottom before every result exists.'''
    p = Path(path)
    if not p.exists():
        print(f'[pending] {p} not found -- run the producing step on adroit.')
        return None
    return loader(p)

# Part A - full CUB-200

## A1. Dataset summary

In [ ]:
rows = []
for split in ('train','val','test'):
    recs = maybe(DATA/'CUB_processed'/'class_attr_data_10'/f'{split}.pkl', lambda p: pd.read_pickle(p))
    if recs is None: continue
    A = np.array([r['attribute_label'] for r in recs])
    rows.append({'split':split,'n_images':len(recs),'n_classes':len({r['class_label'] for r in recs}),
                 'n_attr':A.shape[1],'prev_min':A.mean(0).min().round(3),'prev_max':A.mean(0).max().round(3)})
pd.DataFrame(rows)

## A2. Attribute prevalence histogram (~112 attributes)

In [ ]:
recs = maybe(DATA/'CUB_processed'/'class_attr_data_10'/'train.pkl', lambda p: pd.read_pickle(p))
if recs is not None:
    prev = np.array([r['attribute_label'] for r in recs]).mean(0)
    fig, ax = plt.subplots(); ax.hist(prev, bins=30, color=plotting.PALETTE['CBM'])
    ax.set_xlabel('attribute prevalence'); ax.set_ylabel('# attributes'); plotting.savefig('cub_attr_prevalence')

## A3. CBM training curves

In [ ]:
hist = maybe(RUNS/'cub_cbm_seed1'/'history.parquet')
if hist is not None:
    fig, ax = plt.subplots()
    for col in ('concept_loss','task_loss'):
        if col in hist: ax.plot(hist.epoch, hist[col], label=col)
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); plotting.savefig('cub_cbm_curves')

## A4. CBM final metrics

In [ ]:
ev = maybe(RUNS/'cub_cbm_seed1'/'eval_test.parquet', io.load_eval_table)
if ev is not None:
    img = ev.drop_duplicates('image')
    print('task acc', round((img.y_true==img.y_pred).mean(),3),
          '| mean concept acc', round((ev.gt_label==ev.pred_label).mean(),3))

## A5. MCBM training curves (incl. z-regularizer)

In [ ]:
hist = maybe(RUNS/'cub_mcbm_seed42'/'history.parquet')
if hist is not None:
    fig, ax = plt.subplots()
    for col in ('task_loss','concept_loss','z_loss'):
        if col in hist: ax.plot(hist.epoch, hist[col], label=col)
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); plotting.savefig('cub_mcbm_curves')

## A6. MCBM final metrics

In [ ]:
ev_m = maybe(RUNS/'cub_mcbm_seed42'/'eval_test.parquet', io.load_eval_table)
if ev_m is not None:
    img = ev_m.drop_duplicates('image')
    print('task acc', round((img.y_true==img.y_pred).mean(),3),
          '| mean concept acc', round((ev_m.gt_label==ev_m.pred_label).mean(),3))

## A7. CBM vs MCBM bar

In [ ]:
def metrics(ev):
    if ev is None: return (np.nan,np.nan)
    img = ev.drop_duplicates('image')
    return ((img.y_true==img.y_pred).mean(),(ev.gt_label==ev.pred_label).mean())
vals = {'CBM':metrics(ev),'MCBM':metrics(ev_m)}; x=np.arange(2)
fig, ax = plt.subplots()
for i,(k,(ta,ca)) in enumerate(vals.items()):
    ax.bar(x+i*0.35,[ta,ca],0.35,label=k,color=plotting.PALETTE[k])
ax.set_xticks(x+0.175); ax.set_xticklabels(['task acc','concept acc']); ax.set_ylim(0,1); ax.legend()
plotting.savefig('cub_cbm_vs_mcbm')

## A8. z distribution (fixed-point check)

In [ ]:
fig, ax = plt.subplots()
for ev_,k in [(ev,'CBM'),(ev_m,'MCBM')]:
    if ev_ is not None: ax.hist(ev_['z'],bins=60,alpha=0.5,density=True,label=k,color=plotting.PALETTE[k])
ax.axvline(0,color='k',ls='--',lw=0.8); ax.set_xlabel('z'); ax.legend(); plotting.savefig('cub_z_distribution')

## A9. Example real images: predicted vs GT concepts

In [ ]:
if ev is not None:
    ex = ev[ev.image.isin(ev.image.drop_duplicates().head(3))]
    print(ex.pivot_table(index='image', columns='concept_name', values=['gt_label','pred_label']).head())

## A10. Intervention accuracy curve (Koh et al. headline)

In [ ]:
tti = maybe(RUNS/'cub_tti.parquet')
if tti is not None:
    fig, ax = plt.subplots()
    for k,g in tti.groupby('model'):
        ax.plot(g.n_intervened,g.task_acc,marker='o',label=k,color=plotting.PALETTE.get(k))
    ax.set_xlabel('# concepts intervened'); ax.set_ylabel('task acc'); ax.legend(); plotting.savefig('cub_intervention')

# Part B - CUB70 occlusion / relabeling axis

Ground-truth part visibility comes from CUB70 pixel masks. Maps directly
onto the professor's five notes (see `curated/README.md` section 5).

In [ ]:
vis = maybe(DATA/'cub70_visibility.parquet')
diag = maybe(DATA/'cub70_relabel_diagnostics.parquet')
sys.path.insert(0, str(REPO/'curated'/'data'/'cub70'))
from relabel_cub_with_cub70 import coarse_visibility
cvis = coarse_visibility(vis, 0.001) if vis is not None else None

## B11. CUB70 dataset summary (11 parts, mask area)

In [ ]:
if vis is not None:
    s = (vis.groupby('part').agg(n_images=('image_name','nunique'),
         area_frac_mean=('area_frac','mean'), area_frac_median=('area_frac','median'),
         visible_rate=('visible','mean')).round(4))
    display(s)

## B12. Standard CUB label vs CUB70 visibility (relabel diagnostic, prof note #1)

For each part: of the present-labeled concepts, what fraction land on a
near-zero mask (i.e. the label says present but the part is occluded).

In [ ]:
if diag is not None:
    ct = (diag[diag.original_label==1].groupby('part')
          .agg(present_labeled=('coarse_visible','size'),
               occluded_share=('coarse_visible', lambda s: 1-s.mean())).round(3))
    display(ct)
    fig, ax = plt.subplots(); ct['occluded_share'].plot.barh(ax=ax, color=plotting.PALETTE['occluded'])
    ax.set_xlabel('share of present labels on an occluded part'); plotting.savefig('cub70_label_vs_visibility')

## B13. Relabeled concept table (counts flipped present->absent)

In [ ]:
if diag is not None:
    display(occlusion.relabel_flip_summary(diag).round(3))

## B14. z (full-200 model) vs CUB70 visibility, per part (prof note #2)

In [ ]:
ev200 = maybe(RUNS/'cub_cbm_seed1'/'eval_cub70.parquet', io.load_eval_table)
if ev200 is not None and cvis is not None:
    j = occlusion.attach_visibility(ev200, cvis.rename(columns={'coarse':'part'}))
    display(occlusion.z_by_visibility(j).round(3))
    fig, ax = plt.subplots()
    for vflag,lab,col in [(True,'visible','visible'),(False,'occluded','occluded')]:
        d = j[(j.gt_label==1)&(j.visible.astype(bool)==vflag)]
        ax.scatter(d.part, d.prob, s=6, alpha=0.3, label=lab, color=plotting.PALETTE[col])
    ax.set_ylabel('predicted concept prob'); ax.legend(); plt.xticks(rotation=45, ha='right')
    plotting.savefig('cub70_z_vs_visibility_full200')

## B15. CUB70-only training curves

In [ ]:
hist = maybe(RUNS/'cub70_cbm_original_seed1'/'history.parquet')
if hist is not None:
    fig, ax = plt.subplots()
    for col in ('concept_loss','task_loss'):
        if col in hist: ax.plot(hist.epoch, hist[col], label=col)
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend(); plotting.savefig('cub70_train_curves')

## B16. z vs CUB70 visibility on the CUB70-only model (prof note #4)

In [ ]:
ev70 = maybe(RUNS/'cub70_cbm_original_seed1'/'eval_cub70.parquet', io.load_eval_table)
if ev70 is not None and cvis is not None:
    j70 = occlusion.attach_visibility(ev70, cvis.rename(columns={'coarse':'part'}))
    display(occlusion.z_by_visibility(j70).round(3))

## B17. Comparison table across three conditions (concept & task acc)

In [ ]:
ev70r = maybe(RUNS/'cub70_cbm_relabeled_seed1'/'eval_cub70.parquet', io.load_eval_table)
tables = {k:v for k,v in {'full200_on_cub70':ev200,'cub70_original':ev70,
                          'cub70_relabeled':ev70r}.items() if v is not None}
if tables and cvis is not None:
    display(occlusion.condition_comparison(tables, cvis.rename(columns={'coarse':'part'})).round(3))

## B18. Verdict table -- grounding violation rate by condition (prof note #5)

The single table that says whether relabeling alone fixes grounding,
whether restricting training data alone fixes it, or the problem persists
-- i.e. whether the full pixel-segmentation investment is justified.

In [ ]:
if tables and cvis is not None:
    verdict = occlusion.condition_comparison(tables, cvis.rename(columns={'coarse':'part'}))[['condition','violation_rate']]
    display(verdict.round(3))
    fig, ax = plt.subplots(); ax.bar(verdict.condition, verdict.violation_rate, color=plotting.PALETTE['MCBM'])
    ax.set_ylabel('grounding violation rate'); plt.xticks(rotation=20, ha='right'); ax.set_ylim(0,1)
    plotting.savefig('cub70_verdict')